# Deepfake Audio Detection — End-to-End Walkthrough

Classify speech as **Genuine (Human)** or **Deepfake (AI-Generated)**.

This notebook runs the full pipeline using the project's `src/` modules:
**data → log-mel features → CNN → probability**, then evaluation and a live
inference demo.

**Label convention:** `0 = Genuine`, `1 = Deepfake`. The detection score is the
deepfake probability; EER uses *deepfake* as the positive class.

**Verification targets (problem statement §4/§5):** Accuracy ≥ 80 %, EER ≤ 12 %,
F1 ≥ 80 %, per-class accuracy ≥ 75 %, and a reported confusion matrix.

## 1. Setup

If running on Colab/Kaggle, clone the repo and install dependencies. Locally,
just make sure you've run `pip install -r requirements.txt`.

In [ ]:
# --- Colab / Kaggle setup (uncomment if needed) ---
# !git clone https://github.com/<your-username>/deepfake-audio-detection.git
# %cd deepfake-audio-detection
# !pip install -r requirements.txt

# Make sure we're at the repo root (the folder containing `src/`).
import os, sys
if not os.path.isdir("src") and os.path.isdir("../src"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
import numpy as np
import torch

from src.config import Config
from src.utils import seed_everything, get_device, count_parameters

cfg = Config.load("config.yaml")   # falls back to defaults if file missing
seed_everything(cfg.train.seed)
device = get_device("auto")
print("Device:", device)
print("Torch:", torch.__version__)

## 2. Dataset

Download **The Fake-or-Real dataset** from Kaggle
(`kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset`) and arrange
each split with class sub-folders (`real/`, `fake/`, … — many aliases accepted):

```
data/for-norm/{training,validation,testing}/{real,fake}/*.wav
```

The cell below scans the splits and shows the class balance. Adjust
`cfg.paths.*` if your layout differs.

In [ ]:
from src.dataset import scan_dataset, class_distribution

train_samples = scan_dataset(cfg.paths.train_dir)
print("Train distribution:", dict(class_distribution(train_samples)),
      "| total", len(train_samples))

# Validation + test are optional to scan here but useful as a sanity check:
import os
if os.path.isdir(cfg.paths.test_dir):
    test_samples = scan_dataset(cfg.paths.test_dir)
    print("Test  distribution:", dict(class_distribution(test_samples)),
          "| total", len(test_samples))

### Visualise one genuine and one deepfake clip

We plot the waveform and the **log-mel spectrogram** the model actually sees.
Deepfakes often show subtly over-smoothed spectra and unnatural harmonic/phase
structure in this time–frequency view.

In [ ]:
import matplotlib.pyplot as plt
from src.features import load_waveform, audio_path_to_feature

def first_of(label):
    for p, l in train_samples:
        if l == label:
            return p
    return None

paths = {0: first_of(0), 1: first_of(1)}
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for col, (lbl, name) in enumerate([(0, "Genuine"), (1, "Deepfake")]):
    p = paths[lbl]
    if p is None:
        continue
    y = load_waveform(p, cfg.audio)
    feat = audio_path_to_feature(p, cfg.audio, cfg.features)
    axes[0, col].plot(y); axes[0, col].set_title(f"{name} — waveform")
    im = axes[1, col].imshow(feat, origin="lower", aspect="auto", cmap="magma")
    axes[1, col].set_title(f"{name} — log-mel {feat.shape}")
plt.tight_layout(); plt.show()

## 3. Features

Each clip is resampled to 16 kHz mono, fixed to 4 s (loop-pad / centre-crop),
converted to a **128-bin log-mel spectrogram** (`n_fft=1024`, `hop=256`),
then **instance-normalised** (per-clip mean/var). Shape: **(128, 251)**.
**SpecAugment** (frequency/time masking) is applied during training only.

In [ ]:
from src.features import expected_num_frames
print("Feature shape (n_mels, n_frames):",
      (cfg.features.n_mels, expected_num_frames(cfg.audio, cfg.features)))

## 4. Model

`SpecNetCNN` — a compact VGG-style 2-D CNN: four `Conv-BN-ReLU ×2 → MaxPool →
Dropout` blocks (`32→64→128→128`), a global `AdaptiveAvgPool2d`, then a small
MLP head producing two logits. Global pooling makes it length-agnostic.

In [ ]:
from src.model import build_model, predict_proba

model = build_model(cfg.model).to(device)
print(f"Parameters: {count_parameters(model):,}")

# Sanity check: forward a random batch -> (B, 2) logits
dummy = torch.randn(2, 1, cfg.features.n_mels,
                    expected_num_frames(cfg.audio, cfg.features)).to(device)
with torch.no_grad():
    print("Logits shape:", model(dummy).shape)
    print("Probs shape :", predict_proba(model, dummy).shape)

## 5. Train

This calls the same `src.train.train` used by the CLI. For a quick notebook
demo we lower the epoch count; **raise `cfg.train.epochs` (e.g. 30–50) for a
real run** that can meet the thresholds. The best checkpoint (lowest validation
EER) is saved to `models/best_model.pt`.

In [ ]:
from src.train import train

# Demo setting — increase for a real run:
cfg.train.epochs = 5

checkpoint = train(cfg)
print("Best epoch:", checkpoint["best_epoch"],
      "| val EER: %.2f%%" % (checkpoint["val_eer"] * 100),
      "| val acc: %.3f" % checkpoint["val_accuracy"])

In [ ]:
# Learning curves saved during training:
from IPython.display import Image, display
import os
p = os.path.join(cfg.paths.figures_dir, "training_history.png")
if os.path.exists(p):
    display(Image(filename=p))

## 6. Evaluate

Runs the trained model over the **test set** and computes the full metric suite,
saves figures, and writes `reports/performance_report.md` with an explicit
PASS/FAIL against the verification thresholds.

In [ ]:
from src.evaluate import evaluate

report = evaluate(cfg, cfg.paths.model_path)
print("\nPasses verification:", report.passes_verification)

In [ ]:
# Show the saved figures
from IPython.display import Image, display
import os
for fname in ["confusion_matrix.png", "roc_curve.png"]:
    fp = os.path.join(cfg.paths.figures_dir, fname)
    if os.path.exists(fp):
        display(Image(filename=fp))

## 7. Inference demo — test new audio

Load the saved checkpoint through the shared inference API and classify a clip.
This is the same code path the CLI (`python -m src.predict`) and the Streamlit
app use.

In [ ]:
from src.predict import load_bundle, predict_file

bundle = load_bundle(cfg.paths.model_path)

# Point this at any audio file you want to test:
example = first_of(1) or first_of(0)   # reuse a dataset clip for the demo
result = predict_file(bundle, example)
print(result)
print(f"\n{os.path.basename(example)} -> {result['label_name']} "
      f"(confidence {result['confidence']*100:.1f}%)")

## 8. Reproduce / next steps

- **Full training run:** set `cfg.train.epochs` to 30–50 and re-run section 5,
  or from a shell: `python -m src.train`.
- **Evaluate:** `python -m src.evaluate` → updates `reports/performance_report.md`.
- **Test a file:** `python -m src.predict path/to/clip.wav`.
- **Web app:** `streamlit run app/streamlit_app.py`.
- **Improve robustness:** add ASVspoof 2019 spoof data, tune SpecAugment, or
  increase model width/epochs. Model selection is by **validation EER**, the
  metric the brief is strictest on (≤ 12 %).